# Book Exercises 6. Simple linear regression

This notebook contains similar exercises from the book in the style of Newbold, Carlson and Thorne, chapter 11. You write the code yourselves, in the empty cells. Work alone or in a group, and ask the TA when you are stuck.

The ten exercises use real data where possible and invented (but realistic) numbers otherwise:

| Exercise | Topic | Data |
|---|---|---|
| 1 | Petrol price and petrol sales in Denmark, 2026 | Real: EU Weekly Oil Bulletin and the Danish Energy Agency (CSV) |
| 2 | Margarine consumption and divorce rate in Maine | Real: Tyler Vigen, *Spurious correlations* (CSV) |
| 3 | Interpreting a given regression line | Numbers in the text |
| 4 | Workers and units produced at Company X | Invented (CSV) |
| 5 | AI-skills score and hours saved | Invented (in the notebook) |
| 6 | The beta of Apple (AAPL) against the S&P 500 | Real: Yahoo Finance (CSV) |
| 7 | SST, SSR, SSE, $R^2$ and $s_e^2$ for the Exercise 6 model | Same data as Exercise 6 |
| 8 | Promotion spend and sales growth in a retail chain | Invented (CSV) |
| 9 | Confidence and prediction interval from summary numbers | Numbers in the text |
| 10 | Catalyst concentration and curing time in a laboratory | Invented (in the notebook) |

The data files are in the folder `ws6_data/`, next to this notebook. Keep that folder next to the notebook (or change the paths in the Setup cell).

**Rules.** Write inline code only: do not define your own functions. Use the toolbox below. Under each code cell there is a cell **Your answer:** where you write the interpretation in words, with units.

## Toolbox: Python

The functions you need today. Each one has a tiny example. `df` is a pandas DataFrame, `x` and `y` are columns (or arrays) of the same length.

| Function | What it does | Tiny example |
|---|---|---|
| `pd.read_csv(path)` | Reads a CSV file into a DataFrame | `df = pd.read_csv("ws6_data/file.csv")` |
| `df["col"]`, `df.loc[mask, "col"]` | One column, or the column for the rows where a condition holds | `d26 = df[df["year"] == 2026]` |
| `df.head()`, `len(df)`, `df.dropna()` | First rows, number of rows, drop rows with missing values | `n = len(df)` |
| `x.mean()`, `x.std(ddof=1)`, `x.var(ddof=1)` | Mean, standard deviation and variance (dividing by n − 1) | `xbar = x.mean()` |
| `np.sum(a)`, `np.mean(a)`, `a.sum()` | Sum and mean of an array or column | `Sxx = np.sum((x - x.mean())**2)` |
| `np.sqrt(a)` | Square root | `se = np.sqrt(s2e / Sxx)` |
| `np.cov(x, y)` | 2×2 covariance matrix (divides by n − 1); the covariance is `[0, 1]` | `np.cov(x, y)[0, 1]` |
| `np.var(x, ddof=1)` | Variance with n − 1 (the default `ddof=0` divides by n) | `np.var(x, ddof=1)` |
| `np.corrcoef(x, y)` | 2×2 correlation matrix; the correlation r is `[0, 1]` | `r = np.corrcoef(x, y)[0, 1]` |
| `df.corr()` | Correlation matrix of all numeric columns of a DataFrame | `df[["x", "y"]].corr()` |
| `x.pct_change()` | Relative change from the previous row (a return when x is a price) | `ret = price.pct_change().dropna()` |
| `np.polyfit(x, y, 1)` | Least squares line: returns `[slope, intercept]` | `b1, b0 = np.polyfit(x, y, 1)` |
| `stats.linregress(x, y)` | Regression with `.slope`, `.intercept`, `.rvalue`, `.pvalue`, `.stderr`, `.intercept_stderr` | `res = stats.linregress(x, y)` |
| `stats.t.ppf(q, df=...)` | Critical value: the t value with area q below it | `stats.t.ppf(0.975, df=n - 2)` |
| `stats.t.sf(t, df=...)`, `stats.t.cdf(t, df=...)` | Area above t (use it for p-values), area below t | `2 * stats.t.sf(abs(t), df=n - 2)` |
| `np.linspace(a, b, k)` | k equally spaced numbers between a and b (to draw a line) | `xs = np.linspace(x.min(), x.max(), 50)` |
| `plt.scatter(x, y)`, `plt.plot(xs, ys)` | Scatter plot of the points, line through points | `plt.plot(xs, b0 + b1 * xs, color="red")` |
| `plt.xlabel(...)`, `plt.ylabel(...)`, `plt.title(...)`, `plt.show()` | Axis labels, title, display the figure | `plt.xlabel("Price (DKK per litre)")` |
| `print(f"...{value:.3f}")` | Print text with a number rounded to 3 decimals | `print(f"b1 = {b1:.3f}")` |

## Toolbox: formulas

For a sample of $n$ pairs $(x_i, y_i)$:

**Fitted line.** $S_{xx} = \sum (x_i - \bar x)^2$, $S_{xy} = \sum (x_i - \bar x)(y_i - \bar y)$

$$b_1 = \frac{S_{xy}}{S_{xx}}, \qquad b_0 = \bar y - b_1 \bar x, \qquad \hat y = b_0 + b_1 x, \qquad e_i = y_i - \hat y_i$$

**Sums of squares and $R^2$.**

$$SST = \sum (y_i - \bar y)^2, \quad SSR = \sum (\hat y_i - \bar y)^2, \quad SSE = \sum (y_i - \hat y_i)^2, \quad SST = SSR + SSE$$

$$R^2 = \frac{SSR}{SST} = 1 - \frac{SSE}{SST}, \qquad r = \frac{S_{xy}}{\sqrt{S_{xx} S_{yy}}} \quad (R^2 = r^2)$$

**Error variance and standard errors.**

$$s_e^2 = \frac{SSE}{n-2}, \qquad s_e = \sqrt{s_e^2}, \qquad s_{b_1}^2 = \frac{s_e^2}{S_{xx}}, \qquad s_{b_1} = \frac{s_e}{\sqrt{S_{xx}}}$$

**Inference on the slope** ($df = n - 2$).

$$b_1 \pm t_{n-2,\,\alpha/2}\; s_{b_1}, \qquad t = \frac{b_1 - \beta_1^{0}}{s_{b_1}}$$

**Interval at a value $x_0$** (with $\hat y_0 = b_0 + b_1 x_0$).

$$\text{CI for the mean of } y:\quad \hat y_0 \pm t_{n-2,\,\alpha/2}\; s_e \sqrt{\frac{1}{n} + \frac{(x_0 - \bar x)^2}{S_{xx}}}$$

$$\text{Prediction interval for one new } y:\quad \hat y_0 \pm t_{n-2,\,\alpha/2}\; s_e \sqrt{1 + \frac{1}{n} + \frac{(x_0 - \bar x)^2}{S_{xx}}}$$

**Beta in finance.** $\beta = \dfrac{\operatorname{Cov}(R_i, R_m)}{\operatorname{Var}(R_m)}$, which is exactly the slope $b_1$ of the regression of the asset return on the market return (Exercise 6).

## Setup

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
url = "https://raw.githubusercontent.com/aaubs/AQM-2026/main/data/"
oil = pd.read_csv(url + "denmark_oil_price_quantity.csv")        # Exercise 1
marg = pd.read_csv(url + "margarine_divorce_maine.csv")           # Exercise 2
company = pd.read_csv(url + "company_x_production.csv")           # Exercise 4
spx = pd.read_csv(url + "spx_aapl_returns.csv", parse_dates=["date"])  # Exercises 6 and 7
retail = pd.read_csv(url + "retail_promo_sales.csv")              # Exercise 8

---
## Exercise 1. Petrol price and petrol sales in Denmark, 2026

A fuel-industry analyst wants to know how the monthly sales of petrol in Denmark move with the pump price. The file `denmark_oil_price_quantity.csv` has one row per month:

- `avg_petrol95_price_dkk_per_litre`: the monthly mean of the weekly Danish consumer price of Euro-super 95 petrol, including taxes, in DKK per litre (EU Weekly Oil Bulletin, converted from EUR with the exchange rate of the bulletin).
- `petrol_sales_million_litres`: domestic sales of motor petrol in Denmark in million litres per month (Danish Energy Agency, monthly oil statistics, sheet *Motorbenzin*; includes the bioethanol blended into petrol; border trade is not included).

The data cover **January to September 2026**, so $n = 9$ months. The file also holds the 21 months from January 2025, which you use in part (e). Let $x$ = price and $y$ = quantity sold.

a. Make a scatter plot of quantity against price for the 2026 months.  
b. Compute $b_1$ (and $r$) and interpret $b_1$.  
c. Compute $b_0$ and interpret it. Is the interpretation meaningful?  
d. Predict the monthly sales at a price of 17 DKK per litre. Is 17 inside the range of the data?  
e. (Extension) Repeat the fit with all 21 months from January 2025 to September 2026. What changes? Why is a regression of monthly sales on price on this small sample not a demand curve?

In [ ]:
# Hint: keep the rows with year == 2026, then x = price column, y = quantity column.
# plt.scatter(x, y), then plt.xlabel / plt.ylabel / plt.title / plt.show()

**Your answer:**



In [ ]:
# Hint: Sxx and Sxy from the toolbox, b1 = Sxy / Sxx.
# Check your b1 with stats.linregress(x, y).slope and np.corrcoef(x, y)[0, 1] for r.

**Your answer:**



In [ ]:
# Hint: b0 = ybar - b1 * xbar (from the previous cell). Print the range of x with x.min() and x.max().

**Your answer:**



In [ ]:
# Hint: yhat = b0 + b1 * 17. Compare 17 with x.min() and x.max().

**Your answer:**



In [ ]:
# Hint: use all rows of oil. stats.linregress(...) gives slope, intercept, rvalue, pvalue.
# Plot 2025 and 2026 with different colours (mask with oil["year"] == 2025) and add the fitted line.

**Your answer:**



---
## Exercise 2. Margarine consumption and divorces

One of the best-known *spurious correlations* (Tyler Vigen, *Spurious Correlations*) links **per-capita margarine consumption in the United States** (pounds per person per year, USDA) to the **divorce rate in the US state of Maine** (divorces per 1,000 people, CDC/NVSS). The file `margarine_divorce_maine.csv` has the ten years 2000 to 2009. Let $x$ = margarine consumption and $y$ = Maine divorce rate.

a. Make a scatter plot of the divorce rate against margarine consumption.  
b. Compute $b_0$, $b_1$ and the correlation $r$. Interpret $b_1$.  
c. Does the regression show that eating margarine causes divorces (or that divorces make people eat less margarine)? Explain why a strong correlation does not imply causation, and give the range of $x$ where the line may be used.  
d. (Extension) Compute the correlation of each variable with the year. What does this tell you about the common cause of the two series?

In [ ]:
# Hint: x = margarine column, y = divorce column; plt.scatter(x, y) with axis labels.

**Your answer:**



In [ ]:
# Hint: b1 = Sxy / Sxx, b0 = ybar - b1 * xbar (or stats.linregress); r = np.corrcoef(x, y)[0, 1].
# Draw the line with np.linspace and plt.plot.

**Your answer:**



**c. Does the regression show causation?** (write your answer in the cell below)

**Your answer:**



In [ ]:
# Hint: np.corrcoef(marg["year"], x)[0, 1] and the same for y.

**Your answer:**



---
## Exercise 3. Interpreting a given regression line

A bakery chain regresses the **monthly sales of a branch** ($y$, in DKK thousand) on its **monthly spending on social-media advertising** ($x$, in DKK thousand). The estimated line, from 25 branch-months with $x$ between 5 and 12, is

$$\hat y = 250 + 15x$$

a. Interpret $b_1$ (and say whether $b_0$ has a practical meaning).  
b. By how much does the model predict sales to change if advertising spending increases by 4 (thousand DKK)? If it decreases by 6?  
c. Predict sales for $x = 8$ and for $x = 20$. Which prediction do you trust more, and why?  
d. Does the regression prove that raising advertising spending *causes* higher sales? Explain.

In [ ]:
# Hint: b1 is the change in y-hat for a 1-unit change in x. x = 0 is outside the data (5 to 12).

**Your answer:**



In [ ]:
# Hint: change in y-hat = b1 * (change in x).

**Your answer:**



In [ ]:
# Hint: y-hat = b0 + b1 * x0 for x0 = 8 and x0 = 20. Compare x0 with the range 5 to 12.

**Your answer:**



In [ ]:
# no code needed: answer in words

**Your answer:**



---
## Exercise 4. Workers and production at Company X

The production manager of **Company X**  wants to know how daily output depends on the number of workers on the line. The file `company_x_production.csv` has 22 shifts with two variables: `workers` (number of workers on the line, $x$) and `units_produced` (units produced in the shift, $y$). The data are invented for this workshop.

a. Make a scatter plot of units produced against the number of workers.  
b. Compute $b_0$, $b_1$, the mean number of workers $\bar x$ and the mean output $\bar y$.  
c. Interpret the slope $b_1$ for the manager.  
d. Predict the output for a shift with 30 workers.  
e. The manager asks for the expected output with only 5 workers on the line. Calculate it and say whether you would rely on it.

In [ ]:
# Hint: x = workers, y = units_produced; plt.scatter(x, y) and axis labels.

**Your answer:**



In [ ]:
# Hint: Sxx, Sxy, b1 = Sxy / Sxx, b0 = ybar - b1 * xbar. Check with stats.linregress.

**Your answer:**



In [ ]:
# Hint: b1 = change in y-hat for one extra worker. Say it in units per shift.

**Your answer:**



In [ ]:
# Hint: y-hat = b0 + b1 * 30. Is 30 inside the range of x?

**Your answer:**



In [ ]:
# Hint: y-hat = b0 + b1 * 5. Compare 5 with x.min().

**Your answer:**



---
## Exercise 5. Evaluating AI skills in a company

A company is evaluating the **AI skills** of its employees. Ten employees took an AI-skills assessment (a score from 0 to 100, $x$). Over the next month the company recorded the **hours per week each employee saved** by using AI tools in their daily work ($y$):

| Employee | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 |
|---|---|---|---|---|---|---|---|---|---|---|
| AI-skills score $x$ | 45 | 52 | 58 | 63 | 67 | 72 | 78 | 84 | 89 | 95 |
| Hours saved per week $y$ | 2.4 | 2.1 | 3.6 | 2.9 | 4.4 | 3.8 | 5.6 | 4.7 | 6.4 | 6.1 |

a. Estimate the regression of hours saved on the AI-skills score (find $b_0$ and $b_1$).  
b. Interpret the slope. What would the company expect for an employee whose score is 10 points higher?  
c. (Extension) Predict the hours saved for an employee with a score of 70.

In [ ]:
# Hint: put the two rows of the table in numpy arrays x and y (or lists), then Sxx, Sxy, b1, b0.
# Check with np.polyfit(x, y, 1) or stats.linregress(x, y).

**Your answer:**



In [ ]:
# Hint: the expected difference for 10 points is 10 * b1.

**Your answer:**



In [ ]:
# Hint: y-hat = b0 + b1 * 70 (70 is inside the range 45 to 95).

**Your answer:**



---
## Exercise 6. The beta of Apple against the S&P 500

**What is beta in finance?** In the Capital Asset Pricing Model (CAPM), the **beta** of a stock measures how strongly the stock's return moves with the return of the whole market. If $R_i$ is the return of the stock and $R_m$ the return of the market (here the S&P 500 index), then

$$\beta = \frac{\operatorname{Cov}(R_i, R_m)}{\operatorname{Var}(R_m)}$$

which is exactly the **slope** of the regression $R_i = \alpha + \beta R_m + \varepsilon$. How to read it: $\beta = 1$ means the stock moves, on average, one-for-one with the market; $\beta > 1$ means it is *more volatile* than the market (it amplifies market moves, an "aggressive" stock); $0 < \beta < 1$ is less volatile than the market ("defensive"); $\beta < 0$ moves against the market. 

The file `spx_aapl_returns.csv` has **504 daily returns from 2024-11-04 to 2026-11-06** (two years, trading days): the closing level of the S&P 500 index `^GSPC` (`spx_close`), Apple's *adjusted* close, and the simple daily returns `spx_return` and `aapl_return` (decimal, for example 0.01 = 1 %). Source: Yahoo Finance. The returns were computed as $P_t / P_{t-1} - 1$ (what `pct_change()` does).

Let $x$ = S&P 500 return and $y$ = AAPL return.

a. Make a scatter plot of AAPL returns against S&P 500 returns.  
b. Estimate the regression of AAPL returns on S&P 500 returns: find $b_0$ ($\alpha$) and $b_1$ ($\beta$).  
c. Compute beta a second time as $\operatorname{Cov}(R_i, R_m)/\operatorname{Var}(R_m)$ with `np.cov` and `np.var`, and a third time as $r \cdot s_y / s_x$. Check that you get the same number.  
d. Interpret $\beta$ and the intercept $\alpha$ (also as an annual figure, with 252 trading days). Is $\alpha$ significantly different from 0 at 5 %?  
e. Test $H_0: \beta = 1$ against $H_1: \beta \neq 1$ at 5 %: is Apple more volatile than the market in these data?

In [ ]:
# Hint: x = spx_return column, y = aapl_return column. plt.scatter(x, y, s=8, alpha=0.5)

**Your answer:**



In [ ]:
# Hint: stats.linregress(x, y) gives slope (beta) and intercept (alpha); or Sxy / Sxx and ybar - b1 * xbar.
# Add the fitted line to the scatter plot.

**Your answer:**



In [ ]:
# Hint: np.cov(x, y)[0, 1] / np.var(x, ddof=1)   (use the same ddof in the numerator and denominator).
# Third way: np.corrcoef(x, y)[0, 1] * y.std(ddof=1) / x.std(ddof=1)

**Your answer:**



In [ ]:
# Hint: alpha * 252 for a yearly figure. t = b0 / res.intercept_stderr, p = 2 * stats.t.sf(abs(t), df=n - 2).

**Your answer:**



In [ ]:
# Hint: t = (b1 - 1) / se(b1) with res.stderr, df = n - 2. Two-sided p = 2 * stats.t.sf(abs(t), df).

**Your answer:**



---
## Exercise 7. Standard error of estimate, SST, SSR, SSE and $R^2$ (Exercise 6 data)

Use the same data and regression as in **Exercise 6** (AAPL daily returns on S&P 500 daily returns, `spx_aapl_returns.csv`).

a. Compute the fitted values and the residuals. Compute $SST$, $SSR$ and $SSE$ and verify that $SST = SSR + SSE$.  
b. Compute $R^2$ in two ways ($SSR/SST$ and $1 - SSE/SST$) and compare with $r^2$ from `stats.linregress`. Interpret.  
c. Compute the estimated model error variance $s_e^2 = SSE/(n-2)$ and the standard error of estimate $s_e$. Interpret $s_e$.

In [ ]:
# Hint: refit the line (stats.linregress), yhat = b0 + b1 * x, resid = y - yhat.
# SST = sum((y - ybar)**2), SSR = sum((yhat - ybar)**2), SSE = sum(resid**2). Check np.isclose(SST, SSR + SSE).

**Your answer:**



In [ ]:
# Hint: SSR / SST and 1 - SSE / SST should agree with res.rvalue**2.

**Your answer:**



In [ ]:
# Hint: s2_e = SSE / (n - 2); s_e = np.sqrt(s2_e).

**Your answer:**



---
## Exercise 8. Promotion spend and sales growth in a retail chain

A retail company ran different promotional campaigns in ten of its stores last quarter. For each store the file `retail_promo_sales.csv` gives the **percentage increase in promotion spending** compared with the previous quarter ($x$) and the **percentage increase in store sales** ($y$). The numbers are invented for this workshop.

a. Estimate the least squares regression of sales growth on promotion growth.  
b. Find the unbiased estimate of the variance of the slope estimator, $s_{b_1}^2$.  
c. Find a 95 % confidence interval for the slope $\beta_1$ and interpret it.  
d. Test $H_0: \beta_1 = 0$ against $H_1: \beta_1 \neq 0$ at the 10 % significance level.

In [ ]:
# Hint: Sxx, Sxy, b1, b0 as in the toolbox; scatter plus fitted line.

**Your answer:**



In [ ]:
# Hint: SSE = sum((y - yhat)**2); s2_e = SSE / (n - 2); s2_b1 = s2_e / Sxx.

**Your answer:**



In [ ]:
# Hint: t.ppf(0.975, df=n - 2); interval b1 -/+ t * s_b1.

**Your answer:**



In [ ]:
# Hint: t = b1 / s_b1 (H0: beta1 = 0), df = n - 2, two-sided critical value t.ppf(0.95, df=n - 2).

**Your answer:**



---
## Exercise 9. Confidence interval and prediction interval

From a sample of $n = 27$ observations, a simple regression gave the fitted line $\hat y = 20 + 4x$, with $s_e = 8.5$, $\bar x = 10$ and $\sum (x_i - \bar x)^2 = 640$.

a. Find a 95 % confidence interval for the **mean** of $y$ when $x = 15$.  
b. Find a 95 % prediction interval for an **individual** value of $y$ when $x = 15$.  
c. Which interval is wider and why? What happens to the width of the confidence interval at $x = \bar x = 10$?

In [ ]:
# Hint: y-hat = 20 + 4 * 15; t.ppf(0.975, df=n - 2); half-width t * s_e * sqrt(1/n + (x0 - xbar)**2 / Sxx).

**Your answer:**



In [ ]:
# Hint: same as before but with sqrt(1 + 1/n + (x0 - xbar)**2 / Sxx).

**Your answer:**



In [ ]:
# Hint: compare the two half-widths; then put x0 = xbar = 10 in the CI formula: the second term vanishes.

**Your answer:**



---
## Exercise 10. A laboratory developing a product

A laboratory is developing a new coating. The **curing time** ($y$, minutes until the coating is dry to the touch) is believed to depend on the **concentration of a catalyst** ($x$, % by weight). The lab ran 12 trial batches:

| Batch | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 | 12 |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| Catalyst $x$ (%) | 0.5 | 0.5 | 0.8 | 1.0 | 1.2 | 1.2 | 1.5 | 1.8 | 2.0 | 2.2 | 2.5 | 2.5 |
| Curing time $y$ (min) | 82 | 84 | 76 | 76 | 69 | 71 | 67 | 60 | 54 | 46 | 37 | 37 |

a. Estimate the regression of curing time on catalyst concentration. Interpret the slope.  
b. Find a 90 % confidence interval for the slope and interpret it.  
c. A colleague proposes to use the regression to predict the curing time at a catalyst concentration of 5 %. How useful is that prediction?

In [ ]:
# Hint: enter the two rows as arrays, compute Sxx, Sxy, b1, b0; scatter plus fitted line.

**Your answer:**



In [ ]:
# Hint: s_b1 = s_e / sqrt(Sxx) with s_e^2 = SSE / (n - 2); the 90 % interval uses t.ppf(0.95, df=n - 2).

**Your answer:**



In [ ]:
# Hint: y-hat = b0 + b1 * 5. Compare 5 with the range of x in the trials.

**Your answer:**

